In [1]:
!pip install python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Gauresh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Gauresh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

# This automatically finds and reads your .env file
load_dotenv() 

print("Environment variables loaded!")

Environment variables loaded!


In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib
import os
import warnings

# Suppress warnings to keep output clean
warnings.filterwarnings("ignore", category=UserWarning)

# Dummy HR dataset: [Attendance %, Safety Incidents, Manager Score]
data = {
    'attendance_pct': [98, 85, 99, 72, 100, 95, 80, 65, 92, 88],
    'safety_incidents': [0, 1, 0, 3, 0, 0, 2, 4, 0, 1],
    'manager_score': [8.5, 6.0, 9.0, 4.0, 9.5, 8.0, 5.5, 3.0, 7.5, 6.5],
    'pass_probation': [1, 1, 1, 0, 1, 1, 0, 0, 1, 1] # 1 = Pass, 0 = Fail
}

df = pd.DataFrame(data)
X = df[['attendance_pct', 'safety_incidents', 'manager_score']]
y = df['pass_probation']

# Train the Model
print("Training Sentinel-Flow Risk Model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X.values, y) 

# Create the models folder if it doesn't exist, then save the file
os.makedirs('../backend/models', exist_ok=True)
save_path = '../backend/models/sentinel_risk_model.pkl'
joblib.dump(model, save_path)

print(f"✅ Success! Model saved to: {save_path}")

Training Sentinel-Flow Risk Model...
✅ Success! Model saved to: ../backend/models/sentinel_risk_model.pkl


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# 1. Securely load your environment variables from the .env file
load_dotenv()

# 2. Initialize the client (It automatically finds OPENAI_API_KEY in the background!)
client = OpenAI()

def test_bias_filter(raw_review):
    print("Sending to AI for sanitization...\n")
    
    # The system prompt is the "Brain" of this feature
    system_prompt = """You are an enterprise HR compliance AI. 
    Your job is to read a manager's performance review and rewrite it to be entirely objective, legally safe, and professional. 
    Strip out any emotional language, bias, or personal attacks. 
    Keep only the core performance facts."""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini", # Fast, cheap, and perfect for this task
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": raw_review}
        ],
        temperature=0.2 # Low temperature keeps the AI strictly professional
    )
    return response.choices[0].message.content.strip()

# 3. Let's test it with a highly biased, subjective dummy review
toxic_review = "He is incredibly lazy and always looks annoyed when I ask him to do his job. He showed up 15 minutes late twice this week, which proves he doesn't care about this company at all. His output on the assembly line is fine, but his attitude is completely toxic."

print("🔴 ORIGINAL REVIEW:")
print(toxic_review)
print("\n--------------------------------------------------\n")

clean_review = test_bias_filter(toxic_review)

print("🟢 SANITIZED HR DOCUMENT:")
print(clean_review)

🔴 ORIGINAL REVIEW:
He is incredibly lazy and always looks annoyed when I ask him to do his job. He showed up 15 minutes late twice this week, which proves he doesn't care about this company at all. His output on the assembly line is fine, but his attitude is completely toxic.

--------------------------------------------------

Sending to AI for sanitization...



RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}